# Notebook 03 — Final Evaluation & Analysis
**Assignment 04 | Track 2, Option D | NLP with Deep Learning — IBA**

**What this notebook produces:**
1. **3-way comparison table:** base / sft_A (Crownelius) / sft_B (Nemotron) — all using Qwen3-0.6B
2. **Cross-domain eval:** sft_A model on math prompts, sft_B model on general prompts
3. **Difficulty-stratified eval:** sft_A scored by difficulty tier (easy/medium/hard)
4. **Final summary tables** ready to paste into report

**Prerequisites:** Notebooks 00, 01, 02 must have been run and their CSV/JSON outputs saved.

In [ ]:
!pip install -q "unsloth[cu124-torch260] @ git+https://github.com/unslothai/unsloth.git" 2>/dev/null || \
 pip install -q unsloth
!pip install -q transformers peft datasets accelerate bitsandbytes huggingface_hub pandas python-dotenv

In [ ]:
import os, json, time, re, torch, pandas as pd
from datasets import load_dataset
from unsloth import FastLanguageModel
from huggingface_hub import InferenceClient
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN    = os.getenv("HF_TOKEN")   # loaded from .env
BASE_MODEL  = "Qwen/Qwen3-0.6B"
JUDGE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_SEQ_LEN = 2048

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# All JSON/CSV files are in the same Kaggle working directory
with open("best_trial_datasetA.json") as f:
    best_A = json.load(f)
with open("best_trial_datasetB.json") as f:
    best_B = json.load(f)
with open("prompts.json") as f:
    PROMPTS = json.load(f)
with open("gold_answers.json") as f:
    gold_answers = json.load(f)

print(f"Best Dataset A trial: {best_A['trial']} | adapter: {best_A['adapter_path']}")
print(f"Best Dataset B trial: {best_B['trial']} | adapter: {best_B['adapter_path']}")
print(f"\nBoth adapters are on top of: {BASE_MODEL}")

## Helper Functions

In [ ]:
JUDGE_PROMPT_TEMPLATE = """\
You are an expert evaluator of reasoning quality.

QUESTION:
{question}

REFERENCE ANSWER (Gold):
{gold}

CANDIDATE RESPONSE:
{response}

Evaluate the candidate response on REASONING QUALITY:
5 = Correct answer, clear step-by-step reasoning, no errors
4 = Correct answer, reasoning mostly clear with minor gaps
3 = Partially correct or reasoning has notable gaps
2 = Wrong answer but shows some reasoning attempt
1 = Wrong answer, no meaningful reasoning

Respond in this exact format (nothing else):
SCORE: <number 1-5>
REASON: <one sentence justification>"""


def _parse_judge_output(out):
    score_match = re.search(r"SCORE\s*:\s*([1-5])", out, re.IGNORECASE)
    reason_match = re.search(r"REASON\s*:\s*(.+)", out, re.IGNORECASE | re.DOTALL)
    if not score_match:
        raise ValueError(f"Judge response did not include SCORE 1-5. Raw output: {out!r}")
    reason = reason_match.group(1).strip() if reason_match else out.strip()
    return int(score_match.group(1)), reason


def judge_response(question, gold, response):
    client = InferenceClient(JUDGE_MODEL, token=HF_TOKEN)
    prompt = JUDGE_PROMPT_TEMPLATE.format(question=question, gold=gold, response=response)
    mistral_prompt = f"<s>[INST] {prompt} [/INST]"
    try:
        out = client.text_generation(
            mistral_prompt,
            max_new_tokens=120,
            do_sample=False,
            return_full_text=False,
        ).strip()
        return _parse_judge_output(out)
    except Exception as text_error:
        try:
            out = client.chat_completion(
                messages=[{"role": "user", "content": prompt}],
                max_tokens=120,
                temperature=0.0,
            ).choices[0].message.content.strip()
            return _parse_judge_output(out)
        except Exception as chat_error:
            return 0, f"ERROR: text_generation failed: {text_error}; chat_completion failed: {chat_error}"



def _apply_chat_template(tokenizer, messages):
    """Handles BatchEncoding (transformers 4.50+) and plain tensor (older)."""
    encoded = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    )
    return encoded["input_ids"] if not isinstance(encoded, torch.Tensor) else encoded


def run_prompts(model, tokenizer, prompts, system_msg="You are a helpful reasoning assistant. Think step by step."):
    """Run a list of prompts through a model, return list of response strings."""
    responses = []
    for p in prompts:
        messages = [{"role": "system", "content": system_msg},
                    {"role": "user",   "content": p["prompt"]}]
        input_ids = _apply_chat_template(tokenizer, messages).to(model.device)
        with torch.no_grad():
            output = model.generate(
                input_ids, max_new_tokens=512, temperature=0.1,
                do_sample=True, pad_token_id=tokenizer.eos_token_id
            )
        resp = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        responses.append(resp)
    return responses


def score_responses(prompts, responses, label):
    """Score a list of responses with judge. Returns list of dicts."""
    records = []
    for p, resp in zip(prompts, responses):
        score, reason = judge_response(p["prompt"], gold_answers[p["id"]], resp)
        records.append({"label": label, "prompt_id": p["id"],
                         "type": p["type"], "score": score,
                         "reason": reason, "response": resp})
        print(f"  [{label}] {p['id']}: {score}")
        time.sleep(0.5)
    return records

## Section 1 — 3-Way Comparison Table

Three conditions on all 10 prompts, same model throughout:
- `base` — Qwen3-0.6B with no fine-tuning (from baseline_results.csv)
- `sft_A` — best Dataset A adapter (Crownelius general reasoning)
- `sft_B` — best Dataset B adapter (Nemotron math)

In [ ]:
# Load baseline scores (notebook 00 output) — filter to Qwen3-0.6B only
df_baseline = pd.read_csv("baseline_results.csv")
df_base = df_baseline[df_baseline["model"].str.contains("0.6B")].copy()
df_base["label"] = "base"

all_eval_records = df_base[["label","prompt_id","type","score","reason","response"]].to_dict("records")
print(f"Loaded {len(all_eval_records)} baseline rows for Qwen3-0.6B.")

In [ ]:
# sft_A — best Dataset A adapter
print("Loading sft_A (Crownelius reasoning adapter)...")
model, tokenizer = load_sft_model(best_A["adapter_path"])
responses_A = run_prompts(model, tokenizer, PROMPTS)
records_A   = score_responses(PROMPTS, responses_A, "sft_A")
all_eval_records.extend(records_A)
del model
torch.cuda.empty_cache()
print("Done with sft_A.")

In [ ]:
# sft_B — best Dataset B adapter
print("Loading sft_B (Nemotron math adapter)...")
model_B, tokenizer_B = load_sft_model(best_B["adapter_path"])
responses_B = run_prompts(model_B, tokenizer_B, PROMPTS)
records_B   = score_responses(PROMPTS, responses_B, "sft_B")
all_eval_records.extend(records_B)
# keep model_B alive for cross-domain eval
print("Done with sft_B.")

In [ ]:
df_eval = pd.DataFrame(all_eval_records)

pivot = df_eval.pivot_table(
    index="prompt_id", columns="label", values="score"
)[["base", "sft_A", "sft_B"]]

pivot.loc["AVERAGE"] = pivot.mean().round(2)

print("\n── 3-Way Comparison (Judge Scores 1–5) ──")
print("Model: Qwen3-0.6B | base vs sft_A (Crownelius) vs sft_B (Nemotron)")
print(pivot.to_string())

pivot.to_csv("comparison_3way.csv")
print("\nSaved: comparison_3way.csv")

## Section 2 — Cross-Domain Evaluation

**Question:** Does a model trained on one domain transfer to the other?
- Best 0.6B (trained on general reasoning) → tested on 5 MATH prompts only
- Best 1.7B (trained on math) → tested on 5 GENERAL prompts only

In [ ]:
general_prompts = [p for p in PROMPTS if p["type"] == "general"]
math_prompts    = [p for p in PROMPTS if p["type"] == "math"]

cross_records = []

# sft_A (general-trained) on math prompts
print("Cross-domain: sft_A (general-trained) on MATH prompts")
model_A2, tokenizer_A2 = load_sft_model(best_A["adapter_path"])
resp_cross_A = run_prompts(model_A2, tokenizer_A2, math_prompts)
rec_cross_A  = score_responses(math_prompts, resp_cross_A, "sft_A_on_math")
cross_records.extend(rec_cross_A)
del model_A2, tokenizer_A2
torch.cuda.empty_cache()

# sft_B (math-trained) on general prompts
print("\nCross-domain: sft_B (math-trained) on GENERAL prompts")
resp_cross_B = run_prompts(model_B, tokenizer_B, general_prompts)
rec_cross_B  = score_responses(general_prompts, resp_cross_B, "sft_B_on_general")
cross_records.extend(rec_cross_B)
del model_B, tokenizer_B
torch.cuda.empty_cache()

In [ ]:
df_cross = pd.DataFrame(cross_records)

sft_A_on_math    = df_cross[df_cross["label"] == "sft_A_on_math"]["score"].mean()
sft_B_on_general = df_cross[df_cross["label"] == "sft_B_on_general"]["score"].mean()
sft_A_in_domain  = df_eval[df_eval["label"].eq("sft_A") & df_eval["type"].eq("general")]["score"].mean()
sft_B_in_domain  = df_eval[df_eval["label"].eq("sft_B") & df_eval["type"].eq("math")]["score"].mean()

print("\n── Cross-Domain Transfer (Qwen3-0.6B) ──")
print(f"{'Model':<30} {'In-domain':>12} {'Cross-domain':>14}")
print("-" * 58)
print(f"{'sft_A (general-trained)':<30} {sft_A_in_domain:>12.2f} {sft_A_on_math:>14.2f}  (on math)")
print(f"{'sft_B (math-trained)':<30} {sft_B_in_domain:>12.2f} {sft_B_on_general:>14.2f}  (on general)")

df_cross.to_csv("cross_domain_eval.csv", index=False)
print("\nSaved: cross_domain_eval.csv")

## Section 3 — Difficulty-Stratified Evaluation (Dataset A Best Model)

Dataset A (`Crownelius`) has a `difficulty` column. We evaluate the best 0.6B model
across easy / medium / hard problems to see if SFT improvement is uniform.

In [ ]:
ds_A = load_dataset("Crownelius/Opus-4.6-Reasoning-3300x", split="train")

# Check difficulty values
difficulty_vals = set(ds_A["difficulty"])
print(f"Difficulty values in dataset: {difficulty_vals}")

# Sample 15 per difficulty tier (or all if fewer)
SAMPLE_PER_TIER = 15
diff_samples = {}
for diff in difficulty_vals:
    tier = ds_A.filter(lambda x: x["difficulty"] == diff)
    n    = min(SAMPLE_PER_TIER, len(tier))
    diff_samples[diff] = tier.shuffle(seed=42).select(range(n))
    print(f"  {diff}: {len(tier)} total → using {n}")

In [ ]:
print("Loading best Dataset A model for difficulty-stratified eval...")
model_A3, tokenizer_A3 = load_sft_model(best_A["adapter_path"])

diff_records = []
for diff, tier_ds in diff_samples.items():
    print(f"\n  Difficulty: {diff} ({len(tier_ds)} samples)")
    for row in tier_ds:
        problem  = row["problem"]
        gold     = row["solution"]
        messages = [
            {"role": "system",  "content": "You are a helpful reasoning assistant. Think step by step."},
            {"role": "user",    "content": problem}
        ]
        input_ids = _apply_chat_template(tokenizer_A3, messages).to(model_A3.device)
        with torch.no_grad():
            output = model_A3.generate(
                input_ids, max_new_tokens=512, temperature=0.1,
                do_sample=True, pad_token_id=tokenizer_A3.eos_token_id
            )
        response = tokenizer_A3.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        score, reason = judge_response(problem, gold, response)
        diff_records.append({
            "difficulty": diff, "category": row.get("category", ""),
            "score": score, "reason": reason
        })
        time.sleep(0.3)

del model_A3
torch.cuda.empty_cache()

In [ ]:
df_diff = pd.DataFrame(diff_records)

print("\n── Difficulty-Stratified Results (sft_0.6B on Dataset A problems) ──")
diff_summary = df_diff.groupby("difficulty")["score"].agg(["mean","count"]).round(2)
diff_summary.columns = ["avg_score", "n_samples"]
print(diff_summary.to_string())

df_diff.to_csv("difficulty_eval.csv", index=False)
print("\nSaved: difficulty_eval.csv")

## Section 4 — Final Report Summary Tables

In [ ]:
# Load trial results from notebooks 01 and 02
df_trialsA = pd.read_csv("trial_results_datasetA.csv")
df_trialsB = pd.read_csv("trial_results_datasetB.csv")

print("\n══════════════════════════════════════════════════════")
print(" REPORT TABLE 1 — Dataset A (Qwen3-0.6B) Trial Results")
print("══════════════════════════════════════════════════════")
cols = ["trial","data_pct","rank","lr","epochs","val_loss","avg_judge_score"]
print(df_trialsA[cols].to_string(index=False))
print(f"\nBEST: {best_A['trial']} (score={best_A['avg_judge_score']}, val_loss={best_A['val_loss']})")

print("\n══════════════════════════════════════════════════════")
print(" REPORT TABLE 2 — Dataset B (Qwen3-0.6B) Trial Results")
print("══════════════════════════════════════════════════════")
print(df_trialsB[cols].to_string(index=False))
print(f"\nBEST: {best_B['trial']} (score={best_B['avg_judge_score']}, val_loss={best_B['val_loss']})")

print("\n══════════════════════════════════════════════════════")
print(" REPORT TABLE 3 — 3-Way Comparison (base / sft_A / sft_B)")
print("══════════════════════════════════════════════════════")
print(pd.read_csv("comparison_3way.csv").set_index("prompt_id").to_string())

print("\n══════════════════════════════════════════════════════")
print(" REPORT TABLE 4 — Cross-Domain Transfer")
print("══════════════════════════════════════════════════════")
print(f"sft_A (general-trained) on MATH prompts:    avg = {sft_A_on_math:.2f}")
print(f"sft_B (math-trained)    on GENERAL prompts: avg = {sft_B_on_general:.2f}")

print("\n══════════════════════════════════════════════════════")
print(" REPORT TABLE 5 — Difficulty-Stratified (sft_A on Dataset A problems)")
print("══════════════════════════════════════════════════════")
print(diff_summary.to_string())

print("\nAll output files saved. Ready for report writing.")